In [0]:
%run ./common_utils

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

CAT = "machine_health.prod"
AUDIT = f"{CAT}.pipeline_run_audit"

w = WorkspaceClient()

# ============================================================
# Get job/task parameters
# ============================================================

job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")
task_run_id = dbutils.widgets.get("task_run_id")
machine_type = dbutils.widgets.get("machine_type")

# ============================================================
# Resolve parent job run
# ============================================================

if task_run_id:

    try:
        current_task = w.jobs.get_run(int(task_run_id))

        if current_task.job_run_id:
            job_run_id = str(current_task.job_run_id)

        if current_task.job_id:
            job_id = str(current_task.job_id)

    except Exception as e:
        print(f"Task-run lookup failed: {e}")


# ============================================================
# Fallback if job_run_id is empty
# ============================================================

if not job_run_id:

    fallback = spark.table(AUDIT).where(
        F.col("job_run_id").isNotNull() & (F.col("job_run_id") != "")
    )

    if job_id:
        fallback = fallback.where(F.col("job_id") == job_id)

    if machine_type:
        fallback = fallback.where(F.col("machine_type") == machine_type)

    candidates = (
        fallback
        .select("job_run_id", "start_time")
        .orderBy(F.col("start_time").desc())
        .limit(1)
        .collect()
    )

    if candidates:
        job_run_id = str(candidates[0]["job_run_id"])


if not job_run_id:
    raise ValueError(
        "Could not determine job_run_id."
    )


# ============================================================
# Get parent Job Run
# ============================================================

parent = w.jobs.get_run(int(job_run_id))

print(f"job_id     = {job_id}")
print(f"job_run_id = {job_run_id}")
print(f"run_url    = {parent.run_page_url}")


# ============================================================
# Read audit rows
# ============================================================

rows = (
    spark.table(AUDIT)
    .where(F.col("job_run_id") == job_run_id)
    .collect()
)

print(f"Audit rows = {len(rows)}")


# ============================================================
# Get metadata already captured by audited()
# ============================================================

workspace_id = None
job_name = None
asset_type = machine_type

for r in rows:

    if r.workspace_id:
        workspace_id = r.workspace_id

    if r.job_name:
        job_name = r.job_name

    if r.asset_type:
        asset_type = r.asset_type


# ============================================================
# Enrich each existing audit record
#
# IMPORTANT:
# Update by job_run_id + task_key + attempt_number,
# rather than only task_run_id.
# ============================================================

for r in rows:

    if not r.task_run_id:
        continue

    try:

        task_run = w.jobs.get_run(
            int(r.task_run_id)
        )

        task_url = task_run.run_page_url

        cluster_id = None

        if task_run.cluster_instance:
            cluster_id = (
                task_run
                .cluster_instance
                .cluster_id
            )

        conflicting_url = None

        if r.conflicting_job_run_id:

            try:
                conflicting_run = w.jobs.get_run(
                    int(r.conflicting_job_run_id)
                )

                conflicting_url = (
                    conflicting_run.run_page_url
                )

            except Exception as e:

                print(
                    f"Could not resolve conflicting "
                    f"run {r.conflicting_job_run_id}: {e}"
                )

        print(
            f"Updating {r.task_key} "
            f"attempt {r.attempt_number}"
        )

        spark.sql(
            f"""
            UPDATE {AUDIT}
            SET
                run_page_url = :run_url,
                task_run_page_url = :task_url,
                cluster_id = :cluster_id,
                conflicting_run_page_url = :conflict_url
            WHERE
                job_run_id = :job_run_id
                AND task_key = :task_key
                AND attempt_number = :attempt_number
            """,
            args={
                "run_url": parent.run_page_url,
                "task_url": task_url,
                "cluster_id": cluster_id,
                "conflict_url": conflicting_url,
                "job_run_id": job_run_id,
                "task_key": r.task_key,
                "attempt_number": r.attempt_number
            }
        )

    except Exception as e:

        print(
            f"Could not enrich "
            f"{r.task_key}: {e}"
        )


# ============================================================
# Find tasks already audited
# ============================================================

seen = {
    r.task_key
    for r in rows
    if r.task_key
}


# ============================================================
# Add missing UPSTREAM_FAILED tasks
# ============================================================

for task in parent.tasks:

    result_state = (
        task.state.result_state.value
        if task.state and task.state.result_state
        else None
    )

    if (
        task.task_key not in seen
        and result_state == "UPSTREAM_FAILED"
    ):

        print(
            f"Adding UPSTREAM_FAILED: "
            f"{task.task_key}"
        )

        spark.sql(
            f"""
            INSERT INTO {AUDIT}
            (
                workspace_id,
                job_name,
                asset_type,
                job_id,
                job_run_id,
                task_key,
                machine_type,
                attempt_number,
                status,
                error_message,
                run_page_url
            )
            VALUES
            (
                :workspace_id,
                :job_name,
                :asset_type,
                :job_id,
                :job_run_id,
                :task_key,
                :machine_type,
                1,
                'UPSTREAM_FAILED',
                'Task skipped because an upstream task failed',
                :run_page_url
            )
            """,
            args={
                "workspace_id": workspace_id,
                "job_name": job_name,
                "asset_type": asset_type,
                "job_id": job_id,
                "job_run_id": job_run_id,
                "task_key": task.task_key,
                "machine_type": machine_type,
                "run_page_url": parent.run_page_url
            }
        )


print("collect_run_audit completed successfully.")

job_id     = 
job_run_id = 503576987073750
run_url    = https://dbc-3ae3d30d-6c76.cloud.databricks.com/?o=7474654903538554#job/258091358977862/run/503576987073750
Audit rows = 5
Updating apply_health_rules attempt 1
Updating apply_health_rules attempt 2
Updating build_maintenance_advice attempt 1
Updating build_thermal_advice attempt 1
Updating score_readings attempt 1
collect_run_audit completed successfully.
